# Violence Detection - Two-Stream Architecture
## 🏆 ResNet50 + Optical Flow + LSTM (93-96% Accuracy)
Architecture optimale: RGB Stream + Flow Stream fusionnés avec LSTM

## 🚀 GPU Configuration

In [ ]:
import tensorflow as tf
import os

print('=' * 60)
print('GPU CONFIGURATION')
print('=' * 60)

# Check TensorFlow version
print(f'\nTensorFlow version: {tf.__version__}')

# List all physical devices
print('\n--- Physical Devices ---')
physical_devices = tf.config.list_physical_devices()
for device in physical_devices:
    print(f'  {device.device_type}: {device.name}')

# Check GPU availability
gpus = tf.config.list_physical_devices('GPU')
print(f'\n--- GPU Detection ---')
print(f'Number of GPUs available: {len(gpus)}')

if len(gpus) > 0:
    print('\n✓ NVIDIA GPU detected!')
    
    # Display GPU details
    for i, gpu in enumerate(gpus):
        print(f'\nGPU {i}: {gpu.name}')
        
        # Get GPU details using nvidia-smi if available
        try:
            import subprocess
            result = subprocess.run(
                ['nvidia-smi', '--query-gpu=name,memory.total,driver_version,compute_cap', '--format=csv,noheader'],
                capture_output=True, text=True, timeout=5
            )
            if result.returncode == 0:
                gpu_info = result.stdout.strip().split('\n')[i] if i < len(result.stdout.strip().split('\n')) else ''
                if gpu_info:
                    parts = gpu_info.split(', ')
                    print(f'  Name: {parts[0] if len(parts) > 0 else "Unknown"}')
                    print(f'  Memory: {parts[1] if len(parts) > 1 else "Unknown"}')
                    print(f'  Driver: {parts[2] if len(parts) > 2 else "Unknown"}')
                    print(f'  Compute Capability: {parts[3] if len(parts) > 3 else "Unknown"}')
        except Exception as e:
            print(f'  (nvidia-smi not available: {e})')
    
    # Configure GPU memory growth to prevent OOM errors
    print('\n--- Configuring GPU Memory Growth ---')
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print('✓ Memory growth enabled (prevents OOM errors)')
    except RuntimeError as e:
        print(f'⚠️ Memory growth must be set before GPUs are initialized')
        print(f'   Error: {e}')
    
    # Optional: Set memory limit (uncomment if needed)
    # memory_limit_mb = 8192  # 8GB
    # try:
    #     tf.config.set_logical_device_configuration(
    #         gpus[0],
    #         [tf.config.LogicalDeviceConfiguration(memory_limit=memory_limit_mb)]
    #     )
    #     print(f'✓ Memory limit set to {memory_limit_mb} MB')
    # except RuntimeError as e:
    #     print(f'⚠️ {e}')
    
    # Verify GPU is being used
    print('\n--- GPU Verification ---')
    print(f'Built with CUDA: {tf.test.is_built_with_cuda()}')
    print(f'GPU available: {tf.test.is_gpu_available()}')
    
    # Test GPU with a simple operation
    print('\n--- Testing GPU with sample operation ---')
    with tf.device('/GPU:0'):
        a = tf.constant([[1.0, 2.0], [3.0, 4.0]])
        b = tf.constant([[1.0, 1.0], [0.0, 1.0]])
        c = tf.matmul(a, b)
        print(f'✓ GPU computation successful!')
        print(f'  Result device: {c.device}')
    
    print('\n' + '=' * 60)
    print('🎯 GPU READY FOR TRAINING!')
    print('=' * 60)
    
else:
    print('\n⚠️ WARNING: No GPU detected!')
    print('Training will run on CPU (much slower)')
    print('\nTo use GPU, ensure:')
    print('  1. NVIDIA GPU drivers are installed')
    print('  2. CUDA Toolkit is installed')
    print('  3. cuDNN is installed')
    print('  4. TensorFlow GPU version is installed: pip install tensorflow[and-cuda]')
    print('\n' + '=' * 60)

## ⚡ Performance Optimizations

In [ ]:
print('=' * 60)
print('PERFORMANCE OPTIMIZATIONS')
print('=' * 60)

# 1. Enable Mixed Precision Training (FP16)
# Speeds up training by 2-3x on modern GPUs (RTX, Tesla T4/V100/A100)
print('\n--- Mixed Precision Training (FP16) ---')
try:
    from tensorflow.keras import mixed_precision
    
    # Check if GPU supports mixed precision
    gpus = tf.config.list_physical_devices('GPU')
    if len(gpus) > 0:
        # Get compute capability
        try:
            import subprocess
            result = subprocess.run(
                ['nvidia-smi', '--query-gpu=compute_cap', '--format=csv,noheader'],
                capture_output=True, text=True, timeout=5
            )
            if result.returncode == 0:
                compute_cap = float(result.stdout.strip().split('\n')[0])
                
                # Mixed precision works best on compute capability >= 7.0 (Volta, Turing, Ampere, Ada)
                if compute_cap >= 7.0:
                    mixed_precision.set_global_policy('mixed_float16')
                    print(f'✓ Mixed Precision enabled (FP16)')
                    print(f'  Compute Capability: {compute_cap} (Optimal for FP16)')
                    print(f'  Expected speedup: 2-3x faster training')
                    print(f'  Memory savings: ~30-40%')
                else:
                    print(f'⚠️ Compute Capability: {compute_cap}')
                    print(f'  Mixed precision works best on >= 7.0 (Volta/Turing/Ampere)')
                    print(f'  Keeping default FP32 precision')
            else:
                # Enable anyway if nvidia-smi fails
                mixed_precision.set_global_policy('mixed_float16')
                print('✓ Mixed Precision enabled (FP16)')
                print('  (Could not detect compute capability, enabled anyway)')
        except:
            # Enable anyway
            mixed_precision.set_global_policy('mixed_float16')
            print('✓ Mixed Precision enabled (FP16)')
    else:
        print('⚠️ No GPU detected, keeping FP32 precision')
        
    print(f'\nCurrent policy: {mixed_precision.global_policy().name}')
    print(f'Compute dtype: {mixed_precision.global_policy().compute_dtype}')
    print(f'Variable dtype: {mixed_precision.global_policy().variable_dtype}')
    
except Exception as e:
    print(f'⚠️ Could not enable mixed precision: {e}')
    print('Training will use FP32 (still works fine)')

# 2. Enable XLA (Accelerated Linear Algebra) Compilation
# Optimizes computation graphs for 10-15% speedup
print('\n--- XLA Compilation ---')
try:
    tf.config.optimizer.set_jit(True)
    print('✓ XLA (Accelerated Linear Algebra) enabled')
    print('  Optimizes computation graphs')
    print('  Expected speedup: 10-15%')
    print('  Note: First epoch may be slower (compilation time)')
except Exception as e:
    print(f'⚠️ Could not enable XLA: {e}')

# 3. Configure TensorFlow for optimal performance
print('\n--- TensorFlow Optimizations ---')
try:
    # Enable tensor float 32 on Ampere GPUs (RTX 30xx, A100)
    tf.config.experimental.enable_tensor_float_32_execution(True)
    print('✓ TensorFloat-32 enabled (for Ampere GPUs)')
except:
    pass

# Set optimal thread settings
tf.config.threading.set_inter_op_parallelism_threads(0)  # Auto
tf.config.threading.set_intra_op_parallelism_threads(0)  # Auto
print('✓ Thread parallelism set to auto-optimize')

print('\n' + '=' * 60)
print('⚡ PERFORMANCE OPTIMIZATIONS APPLIED!')
print('=' * 60)
print('\nExpected improvements:')
print('  • Mixed Precision (FP16): 2-3x faster on modern GPUs')
print('  • XLA Compilation: 10-15% speedup')
print('  • Memory savings: 30-40% with FP16')
print('  • Total speedup: Up to 3-4x faster training!')
print('=' * 60 + '\n')

## 📦 Import Libraries

In [ ]:
import numpy as np
import pickle
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau, TensorBoard
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from config import *

np.random.seed(42)
tf.random.set_seed(42)

print('✓ Libraries imported successfully')

## 1. Load Two-Stream Data

In [ ]:
with open('preprocessed_data_optical_flow.pkl', 'rb') as f:
    data = pickle.load(f)

X_rgb = data['X_rgb']
X_flow = data['X_flow']
y = data['y']

print(f'RGB: {X_rgb.shape}')
print(f'Flow: {X_flow.shape}')
print(f'Labels: {y.shape}')
print(f'Violence: {np.sum(y == 1)}, Non-Violence: {np.sum(y == 0)}')

In [ ]:
# Split: 70% train, 15% val, 15% test
X_rgb_train, X_rgb_temp, X_flow_train, X_flow_temp, y_train, y_temp = train_test_split(
    X_rgb, X_flow, y, test_size=0.3, random_state=42, stratify=y
)

X_rgb_val, X_rgb_test, X_flow_val, X_flow_test, y_val, y_test = train_test_split(
    X_rgb_temp, X_flow_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print(f'Train: {len(y_train)}, Val: {len(y_val)}, Test: {len(y_test)}')